# Phase 2 — Exploratory Data Analysis  

**StreamFlix — Content Analytics Project**

# Load All cleaned csv files

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick

#load the cleaned dataframes
folder = Path(r"D:\Internship project2")

csv_files = list(folder.glob("*_cleaned.csv"))
dataframes = {file.stem: pd.read_csv(file) for file in csv_files}

display(csv_files)
display(dataframes.keys())

for name, df in dataframes.items():
    display(df.head())

# Monthly viewing volume 

In [ ]:
# A line chart showing number of sessions per month across the timeline. Highlight the busiest months.

if 'titles_cleaned' in dataframes:
    df = dataframes["titles_cleaned"]

# Convert text dates into datetime values
    df["date_added"] = pd.to_datetime(df["date_added"], errors="coerce", dayfirst=True)

# Count sessions (rows) for every month
    monthly_sessions = (df.dropna(subset=["date_added"])
                        .groupby(df.dropna(subset=["date_added"])["date_added"]
                                 .dt.to_period("M")).size().sort_index())

# Find the peak month(s)
    peak_value = monthly_sessions.max()
    busiest_months = monthly_sessions[monthly_sessions == peak_value]

# Plot
    plt.figure(figsize=(13, 6))
    plt.plot(monthly_sessions.index.to_timestamp(), 
             monthly_sessions.values, marker="o", linewidth=2, color="blue", label="Sessions")

    plt.scatter(busiest_months.index.to_timestamp(), busiest_months.values, 
                color="red", s=110, zorder=3, label="Busiest month")

    for month, count in busiest_months.items():
        plt.annotate(f"{month.strftime('%b %Y')}\n{count:,} sessions", (month.to_timestamp(), count), xytext=(0, 12), 
                     textcoords="offset points", ha="center", color="red", fontweight="bold")

    plt.title("Monthly Viewing Volume")
    plt.xlabel("Month")
    plt.ylabel("Number of Sessions")
    plt.grid(axis="y", alpha=0.3)
    plt.legend()
    plt.tight_layout()
    plt.show()

# Monthly Watch Hours Trend 

In [ ]:
# A line chart showing number of sessions per month using watch_duration_min (converted to hours) also Compare year-over-year.

if 'watch_history_cleaned' in dataframes:
    df = dataframes["watch_history_cleaned"]

# Prepare dates and convert minutes to hours
    df["watch_date"] = pd.to_datetime(df["watch_date"])
    df["watch_hours"] = df["watch_duration_min"] / 60
    df["year"] = df["watch_date"].dt.year
    df["month"] = df["watch_date"].dt.month

# Sum watch hours by month and year
    monthly_hours = (df.groupby(["year", "month"], as_index=False)["watch_hours"].sum())

# Compare the latest three years
    years_to_plot = sorted(monthly_hours["year"].unique())[-3:]
    month_labels = ["Jan", "Feb", "Mar", "Apr", "May", "Jun",
                "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]

    plt.figure(figsize=(12, 6))

for year in years_to_plot:
    year_data = monthly_hours[monthly_hours["year"] == year]
    label = f"{year} YTD" if year == max(years_to_plot) else str(year)

    plt.plot(
        year_data["month"],
        year_data["watch_hours"],
        marker="o",
        linewidth=2.2,
        label=label
    )

plt.title("Monthly Watch Hours — Year-over-Year")
plt.xlabel("Month")
plt.ylabel("Watch hours")
plt.xticks(range(1, 13), month_labels)
plt.gca().yaxis.set_major_formatter( mtick.FuncFormatter(lambda value, _: f"{value/1_000_000:.1f}M"
                        if value >= 1_000_000 else f"{value/1_000:.0f}K"))
plt.grid(axis="y", alpha=0.3)
plt.legend(title="Year")
plt.tight_layout()
plt.show()

# Genre-wise watch-hours share 

In [ ]:
# a horizontal bar chart showing which genre earns the most watch time.
if 'titles_cleaned' in dataframes:
    df = dataframes["titles_cleaned"]
    
# sum total_watch_hours by primary_genre and sort in descending order
    genre_watch_time = df.groupby("primary_genre", as_index=False)["total_watch_hours"].sum().sort_values(by="total_watch_hours", ascending=False)
    
    plt.figure(figsize=(10, 6))
    plt.barh(genre_watch_time["primary_genre"], genre_watch_time["total_watch_hours"], color="blue")
    plt.title("Total Watch Hours by Genre")
    plt.xlabel("Watch Hours")
    plt.ylabel("Genre")
    plt.tight_layout()
    plt.show()

# Content Type Split 

In [ ]:
# A Donut Chart showing the share of hours for Movies vs. TV Shows
if 'titles_cleaned' in dataframes:
    df = dataframes["titles_cleaned"]
    
# sum total_watch_hours by type
    type_watch_time = df.groupby("type", as_index=False)["total_watch_hours"].sum().sort_values(by="total_watch_hours", ascending=False)
    
    plt.figure(figsize=(8, 8))
    plt.pie(type_watch_time["total_watch_hours"], labels=type_watch_time["type"], autopct='%1.1f%%', startangle=140, wedgeprops={'width':0.5})
    plt.title("Share of Watch Hours by Type")
    plt.legend(title="Type", loc="upper right")
    plt.axis('equal')
    plt.show()

# Top 10 countries by watch hours 

In [ ]:
# A horizontal bar chart showing the top 10 countries by total watch hours.
if 'titles_cleaned' in dataframes:
    df = dataframes["titles_cleaned"]
    
# sum total_watch_hours by country and sort in descending order
    country_watch_time = df.groupby("country", as_index=False)["total_watch_hours"].sum().sort_values(by="total_watch_hours", ascending=False).head(10)

    plt.figure(figsize=(10, 6))
    plt.barh(country_watch_time["country"], country_watch_time["total_watch_hours"], color="green")
    plt.title("Top 10 Countries by Total Watch Hours")
    plt.xlabel("Watch Hours")
    plt.ylabel("Country")
    plt.tight_layout()
    plt.show()

# Subscriber Plan Distribution 

In [ ]:
# A Pie chart showing Basic vs. Standard vs. Premium subscribers.
if 'subscribers_cleaned' in dataframes:
    df = dataframes["subscribers_cleaned"]

plan_type_count = (df.groupby("plan_type").size().reset_index(name="subscriber_count")
                   .sort_values(by="subscriber_count", ascending=False))

plt.figure(figsize=(8, 8))

wedges, _, _ = plt.pie(plan_type_count["subscriber_count"],
                       labels=plan_type_count["plan_type"],autopct="%1.1f%%",startangle=140)

plt.title("Total Subscribers by Plan Type")
plt.legend(wedges,plan_type_count["plan_type"],title="Plan Type",loc="upper right")
plt.axis("equal")
plt.show()

# Device usage 

In [ ]:
# A Bar chart showing on which devices people watch most.
if 'watch_history_cleaned' in dataframes:
    df = dataframes["watch_history_cleaned"]
    
    # group by device and sum watch_duration_min
    device_watch_time = df.groupby("device", as_index=False)["watch_duration_min"].sum().sort_values(by="watch_duration_min", ascending=False)

    plt.figure(figsize=(10, 6))
    plt.bar(device_watch_time["device"], device_watch_time["watch_duration_min"], color="blue")
    plt.title("Total Watch Time by Device")
    plt.xlabel("Device")
    plt.ylabel("Watch Time (minutes)")
    plt.tight_layout()
    plt.show()

# Age Distribution of subscribers 

In [ ]:
# A histogram showing which age group watches most.
if 'subscribers_cleaned' in dataframes and 'watch_history_cleaned' in dataframes:
    subscribers_df = dataframes["subscribers_cleaned"]
    watch_history_df = dataframes["watch_history_cleaned"]

    # Merge the two dataframes on subscriber_id
    merged_df = pd.merge(watch_history_df, subscribers_df, on="subscriber_id", how="inner")

    # Plot histogram of age groups
    plt.figure(figsize=(10, 6))
    plt.hist(merged_df["age"], bins=range(0, 100, 10), edgecolor='black', alpha=0.7)
    plt.title("Watch Time Distribution by Age Group")
    plt.xlabel("age")
    plt.ylabel("Number of Sessions")
    plt.xticks(range(0, 100, 10))
    plt.tight_layout()
    plt.show()

# Completion rate by genre 

In [ ]:
# A bar chart showing which genres completed.
if 'titles_cleaned' in dataframes:
    df = dataframes["titles_cleaned"]
    
    # group by primary_genre and sum total_watch_hours
    genre_watch_time = df.groupby("primary_genre", as_index=False)["total_watch_hours"].sum().sort_values(by="total_watch_hours", ascending=False)

    plt.figure(figsize=(10, 6))
    plt.bar(genre_watch_time["primary_genre"], genre_watch_time["total_watch_hours"], color="purple")
    plt.title("Total Watch Hours by Genre")
    plt.xlabel("Genre")
    plt.ylabel("Watch Hours")
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()


# Review sentiment breakdown 

In [ ]:
if "reviews_cleaned" in dataframes:
    df = dataframes["reviews_cleaned"]

    # Count reviews in each sentiment category
    sentiment_counts = (df.groupby("sentiment").size().reset_index(name="count"))

    order = ["Positive", "Neutral", "Negative"]
    sentiment_counts["sentiment"] = pd.Categorical(sentiment_counts["sentiment"],categories=order,ordered=True)
    sentiment_counts = sentiment_counts.sort_values("sentiment")

    colors = ["blue", "orange", "red"]

    plt.figure(figsize=(8, 6))
    plt.bar(sentiment_counts["sentiment"],sentiment_counts["count"],color=colors)
    plt.title("Review Sentiment Counts")
    plt.xlabel("Sentiment")
    plt.ylabel("Number of Reviews")
    plt.tight_layout()
    plt.show()